# 0 — Preparação do dataset (transcrição + verificação + export)

**O que faz:** pega as sessões gravadas com o kit (`tools/recording/`), transcreve os
takes longos com faster-whisper, confere se os takes com script foram lidos certo
(WER por item), e exporta o `data/dataset_v1` final.

**GPU:** T4 basta.

**Antes de rodar — ORDEM no Mac, depois subir pro Drive:**
1. gravar com `record.py` (gera `data/raw/<ses>/metadata.jsonl` + wavs)
2. rodar `segment_long.py` LOCAL (gera `data/raw/<ses>/segments/` + `to_transcribe.jsonl`)
3. SÓ ENTÃO subir a pasta `data/raw/` **inteira** (com a subpasta `segments/`) pro Drive
   em `MyDrive/TTS-ptbr-data/raw/`. Se subir antes do passo 2, a transcrição pula
   silenciosamente e o dataset sai sem os monólogos/diálogos.
4. Secrets no painel 🔑 do Colab: `HF_TOKEN` + `GH_TOKEN` (com "Notebook access" ligado).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')

assert userdata.get('GH_TOKEN'), 'configure GH_TOKEN no painel 🔑 do Colab (com Notebook access)'
GH_TOKEN = userdata.get('GH_TOKEN')
# clone fresco; sem 2>/dev/null pra ver erro de auth se houver
![ -d /content/TTS-ptbr/.git ] && (cd /content/TTS-ptbr && git pull) || git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr
%cd /content/TTS-ptbr
assert os.path.isdir('/content/TTS-ptbr/.git'), 'clone falhou — confira o GH_TOKEN'

# linka o áudio do Drive no layout que os scripts esperam
!mkdir -p data && ln -sfn /content/drive/MyDrive/TTS-ptbr-data/raw data/raw
!ls data/raw/

In [ ]:
%%capture
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
!pip -q install faster-whisper==1.1.0 jiwer==3.0.4 soundfile pyloudnorm soxr hf_transfer
# faster-whisper/ctranslate2 4.5+ usa cuDNN 9 via wheel pip; aponta o loader pra ela
# (modo de falha nº1 no Colab: kernel reinicia ao criar WhisperModel em cuda — issue #1236)
import os
try:
    import nvidia.cudnn, pathlib
    _cudnn = str(pathlib.Path(nvidia.cudnn.__file__).parent / 'lib')
    os.environ['LD_LIBRARY_PATH'] = _cudnn + ':' + os.environ.get('LD_LIBRARY_PATH', '')
except Exception as e:
    print('aviso cudnn path:', e)

In [ ]:
# ⚙️ CONFIG
SESSIONS = ['ses01']            # sessões a processar
ASR_MODEL = 'large-v3'          # juiz de transcrição (T4: ok com int8_float16)
WER_FLAG = 0.12                 # acima disso o take com script vai pra lista de revisão

## 1. Transcrever segmentos dos takes longos (monólogo/diálogo)

In [ ]:
import json, pathlib
from faster_whisper import WhisperModel

asr = WhisperModel(ASR_MODEL, device='cuda', compute_type='int8_float16')

for ses in SESSIONS:
    seg_meta = pathlib.Path(f'data/raw/{ses}/segments/to_transcribe.jsonl')
    if not seg_meta.exists():
        print(f'{ses}: sem segmentos (rode segment_long.py local antes) — ok se só gravou frases');
        continue
    rows = [json.loads(l) for l in seg_meta.read_text(encoding='utf-8').splitlines() if l.strip()]
    out_path = seg_meta.with_name('transcribed.jsonl')
    with out_path.open('w', encoding='utf-8') as f:
        for r in rows:
            segs, info = asr.transcribe(r['audio'], language='pt', vad_filter=False)
            r['text'] = ' '.join(s.text.strip() for s in segs).strip()
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
            print(f"  {r['id']}: {r['text'][:80]}")
    print(f'{ses}: {len(rows)} segmentos transcritos → {out_path}')

## 2. Verificar leitura dos takes com script (WER por item)

In [ ]:
import json, pathlib, jiwer

norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
                      jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
review = []
for ses in SESSIONS:
    meta = pathlib.Path(f'data/raw/{ses}/metadata.jsonl')
    rows = [json.loads(l) for l in meta.read_text(encoding='utf-8').splitlines() if l.strip()]
    scripted = [r for r in rows if r['kind'] in ('sentenca', 'emocao', 'sotaque')]
    for r in scripted:
        segs, _ = asr.transcribe(r['audio'], language='pt')
        hyp = ' '.join(s.text.strip() for s in segs).strip()
        wer = jiwer.wer(norm(r['text']), norm(hyp)) if hyp else 1.0
        r['asr_text'], r['asr_wer'] = hyp, round(wer, 3)
        if wer > WER_FLAG:
            review.append((r['id'], wer, r['text'], hyp))
    checked = meta.with_name('metadata_checked.jsonl')
    with checked.open('w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'{ses}: {len(scripted)} takes verificados → {checked}')

print(f'\n⚠️ {len(review)} takes para revisar (leu diferente do script):')
for rid, wer, ref, hyp in review[:30]:
    print(f'  {rid}  wer={wer:.2f}\n    script: {ref}\n    ouvido: {hyp}')

> **Decisão por take divergente:** se o áudio está bom e a divergência é fala real
> (ex.: contração natural), atualize `text` pro que foi FALADO (verbatim vence o
> script). Se foi erro de leitura, regrave na próxima sessão (o id volta pro plano).

## 3. Exportar dataset final (train/val + formatos por modelo)

In [ ]:
!python tools/recording/export_dataset.py --sessions {' '.join(SESSIONS)} \
    --sr 24000 --format canonical csm orpheus
!python tools/recording/qc_report.py --all --out data/dataset_v1/qc_report.md
# espelha o dataset no Drive (o 'ouro' fica fora do git)
!mkdir -p /content/drive/MyDrive/TTS-ptbr-data/dataset_v1 && cp -r data/dataset_v1/* /content/drive/MyDrive/TTS-ptbr-data/dataset_v1/
print('✅ dataset espelhado no Drive')

## 4. (Opcional) subir pro HF privado — o registro do "ouro"

In [ ]:
# from google.colab import userdata
# from huggingface_hub import HfApi
# api = HfApi(token=userdata.get('HF_TOKEN'))
# api.create_repo('pedrocormann/tts-ptbr-dataset-v1', repo_type='dataset', private=True, exist_ok=True)
# api.upload_folder(folder_path='data/dataset_v1', repo_id='pedrocormann/tts-ptbr-dataset-v1', repo_type='dataset')